In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as k

import DAN_code.data_processing as data_proc
import DAN_code.initializers as init
from DAN_code import models

from tensorflow.keras.datasets import mnist

import matplotlib.pyplot as plt
import DAN_code.result_plotting as res_plot
import DAN_code.result_testing as res_test
import time

import os

eps = np.finfo("float32").eps

# %load_ext tensorboard

### Fig. 1: weights learned with the negative-log likelihood loss and P = 25.

In [ ]:
# Seed of random number generation for reproducibility
seed = 76986
np_rng = np.random.default_rng(seed)
np_seed, tf_seed = np_rng.integers(0, 2**63, size = 2)
np_rng = np.random.default_rng(np_seed)
tf.random.set_seed(tf_seed)

# Dataset and label softening
dataset = mnist
softening = 0.1

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_flat_data(dataset, normalize_data = True, softening = softening)

# Loss type (supervised or unsupervised)
loss = "supervised"

number_features = x_train.shape[1]
number_classes = y_train.shape[1]
batch_size = 1000

# Number of epochs of the memorization and splitting phases
number_memorization_epochs = 300
number_splitting_epochs = 0

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 10

# Maximum number of memories, initial number of memories
# and maximum number of splits of splitting steepest descent
max_number_memories = 25
init_number_memories = 25
number_splits = 0

# Initial inverse temperature and inverse temperature regularization
beta_init = 16.
beta_reg = None

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = 1

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
y_hard = np.argmax(y_train, axis = 1)
counts_y = np.bincount(y_hard)
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

# Name of the model and what to record (see DAN_code.models.train_DAN for record options)
name = "DAN"
record = None

if record == "movies":
    try:
        fname = "./Data/Weights/%s_w_with_beta_reg=%s_and_for_movies.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass
    try:
        fname = "./Data/Weights/%s_g_with_beta_reg=%s_and_for_movies.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass

if record == "beta":
    try:
        fname = "./Data/Weights/%s_beta_with_beta_reg=%s.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass

model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features,
                        init_number_memories, max_number_memories, number_classes,
                        number_constraint_iterations, learning_rate, momentum,
                        prior_y, normalize_online = normalize_online)

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

number_memories = init_number_memories
for split in range(number_splits + 1):
    
    model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                             training_phase = "memorization", record = record, verbose = True)
    
    number_memories = model.get_DAN_layer(1).output_size
    max_number_memories = model.get_DAN_layer(1).max_output_size
    
    if number_memories == max_number_memories:
        print("Maximum number of memories reached, cannot split any further.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    elif split == number_splits:
        print("Maximum number of splits reached, splitting will stop.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    else:
        model = models.train_DAN(x_train, y_train, model, number_splitting_epochs, batch_size, patience,
                                 training_phase = "splitting", record = None, verbose = True)
        
        max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
        mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                      max_eigval, batch_size, adjust = True)
        
        if not k.any(mask):
            print("Found no favorable eigendirection to split, splitting will stop.")
            if save_model:
                models.save_DAN(model, beta_reg, number_memories, split)
            break
        
        init.split_memories(model, mask, 0.25)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0]
    g_cur = model.get_DAN_layer(2).get_weights()[0]
    
    w_plot = w_cur[:, : np.minimum(25, number_memories).astype("int")].T
    g_plot = g_cur[: np.minimum(25, number_memories).astype("int")].T
    
    res_plot.plot_images(w_plot)
    
    res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

models.compile_memorization_phase(model)
number_memories = max_number_memories
number_splits = np.ceil(np.log2(max_number_memories / init_number_memories)).astype("int")

model = models.load_DAN(beta_reg, number_memories, number_splits)

# Get hard test labels and predictions
y_hard = np.argmax(y_test, axis = 1)
y_pred = np.argmax(model(x_test), axis = 1)

model.evaluate(tf.convert_to_tensor(x_test, dtype = "float32"),
               tf.convert_to_tensor(y_test, dtype = "float32"))

w = model.get_DAN_layer(1).get_weights()[0]
g = model.get_DAN_layer(2).get_weights()[0]

#res_test.first_neighbor_fidelity(x_test, y_hard, y_pred, w, g)

w_plot = w[:, : 25].T
g_plot = g[: 25].T
res_plot.plot_images(w_plot)
res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

res_test.sort_memories(model, w)

### Fig. 2 and Table 1: weights and performance of DAMs trained with the negative-log likelihood loss and P = 1000.

User-defined parameters common to all the DAMs that are trained to make Fig. 2 and Table 1.

Run this code cell first, then run the next code cell to choose a value of beta.

In [ ]:
# Seed of random number generation for reproducibility
seed = 76986
np_rng = np.random.default_rng(seed)
np_seed, tf_seed = np_rng.integers(0, 2**63, size = 2)
np_rng = np.random.default_rng(np_seed)
tf.random.set_seed(tf_seed)

# Dataset and label softening
dataset = mnist
softening = 0.1

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_flat_data(dataset, normalize_data = True, softening = softening)

# Loss type (supervised or unsupervised)
loss = "supervised"

number_features = x_train.shape[1]
number_classes = y_train.shape[1]
batch_size = 1000

# Number of epochs of the memorization and splitting phases
number_memorization_epochs = 300
number_splitting_epochs = 0

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 10

# Maximum number of memories, initial number of memories
# and maximum number of splits of splitting steepest descent
max_number_memories = 1000
init_number_memories = 1000
number_splits = 0

# Inverse temperature regularization
beta_reg = None

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = 1

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
y_hard = np.argmax(y_train, axis = 1)
counts_y = np.bincount(y_hard)
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

# Name of the model and what to record (see DAN_code.models.train_DAN for record options)
name = "DAN"
record = None

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

Change beta to choose which plot of Fig. 2 or entry of Table 1 to reproduce.

In [ ]:
#beta_init = 6.
#beta_init = 10.
beta_init = 14.
#beta_init = 18.

Run this code cell to train a DAN for the parameters set in the cells above.

In [ ]:
model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features,
                        init_number_memories, max_number_memories, number_classes,
                        number_constraint_iterations, learning_rate, momentum,
                        prior_y, normalize_online = normalize_online)

number_memories = init_number_memories
for split in range(number_splits + 1):
    
    model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                             training_phase = "memorization", record = record, verbose = True)
    
    number_memories = model.get_DAN_layer(1).output_size
    max_number_memories = model.get_DAN_layer(1).max_output_size
    
    if number_memories == max_number_memories:
        print("Maximum number of memories reached, cannot split any further.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    elif split == number_splits:
        print("Maximum number of splits reached, splitting will stop.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    else:
        model = models.train_DAN(x_train, y_train, model, number_splitting_epochs, batch_size, patience,
                                 training_phase = "splitting", record = None, verbose = True)
        
        max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
        mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                      max_eigval, batch_size, adjust = True)
        
        if not k.any(mask):
            print("Found no favorable eigendirection to split, splitting will stop.")
            if save_model:
                models.save_DAN(model, beta_reg, number_memories, split)
            break
        
        init.split_memories(model, mask, 0.25)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0]
    g_cur = model.get_DAN_layer(2).get_weights()[0]
    
    w_plot = w_cur[:, : np.minimum(25, number_memories).astype("int")].T
    g_plot = g_cur[: np.minimum(25, number_memories).astype("int")].T
    
    res_plot.plot_images(w_plot)
    
    res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

models.compile_memorization_phase(model)
number_memories = max_number_memories
number_splits = np.ceil(np.log2(max_number_memories / init_number_memories)).astype("int")

model = models.load_DAN(beta_reg, number_memories, number_splits)

# Get hard test labels and predictions
y_hard = np.argmax(y_test, axis = 1)
y_pred = np.argmax(model(x_test), axis = 1)

model.evaluate(tf.convert_to_tensor(x_test, dtype = "float32"),
               tf.convert_to_tensor(y_test, dtype = "float32"))

w = model.get_DAN_layer(1).get_weights()[0]
g = model.get_DAN_layer(2).get_weights()[0]

#res_test.first_neighbor_fidelity(x_test, y_hard, y_pred, w, g)

w_plot = w[:, : 25].T
g_plot = g[: 25].T
res_plot.plot_images(w_plot)
res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

res_test.sort_memories(model, w)

### Fig. 3: weights learned by supervised learning with the effective loss.

In [ ]:
# Seed of random number generation for reproducibility
seed = 76986
np_rng = np.random.default_rng(seed)
np_seed, tf_seed = np_rng.integers(0, 2**63, size = 2)
np_rng = np.random.default_rng(np_seed)
tf.random.set_seed(tf_seed)

# Dataset and label softening
dataset = mnist
softening = 0.1

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_flat_data(dataset, normalize_data = True, softening = softening)

# Loss type (supervised or unsupervised)
loss = "supervised"

number_features = x_train.shape[1]
number_classes = y_train.shape[1]
batch_size = 1000

# Number of epochs of the memorization and splitting phases
number_memorization_epochs = 300
number_splitting_epochs = 0

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 10

# Maximum number of memories, initial number of memories
# and maximum number of splits of splitting steepest descent
max_number_memories = 2000
init_number_memories = 2000
number_splits = 0

# Initial inverse temperature and inverse temperature regularization
beta_init = 1.
beta_reg = 0.25

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = 1

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
y_hard = np.argmax(y_train, axis = 1)
counts_y = np.bincount(y_hard)
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

# Name of the model and what to record (see DAN_code.models.train_DAN for record options)
name = "DAN"
record = None

if record == "movies":
    try:
        fname = "./Data/Weights/%s_w_with_beta_reg=%s_and_for_movies.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass
    try:
        fname = "./Data/Weights/%s_g_with_beta_reg=%s_and_for_movies.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass

if record == "beta":
    try:
        fname = "./Data/Weights/%s_beta_with_beta_reg=%s.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass

model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features,
                        init_number_memories, max_number_memories, number_classes,
                        number_constraint_iterations, learning_rate, momentum,
                        prior_y, normalize_online = normalize_online)

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

number_memories = init_number_memories
for split in range(number_splits + 1):
    
    model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                             training_phase = "memorization", record = record, verbose = True)
    
    number_memories = model.get_DAN_layer(1).output_size
    max_number_memories = model.get_DAN_layer(1).max_output_size
    
    if number_memories == max_number_memories:
        print("Maximum number of memories reached, cannot split any further.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    elif split == number_splits:
        print("Maximum number of splits reached, splitting will stop.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    else:
        model = models.train_DAN(x_train, y_train, model, number_splitting_epochs, batch_size, patience,
                                 training_phase = "splitting", record = None, verbose = True)
        
        max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
        mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                      max_eigval, batch_size, adjust = True)
        
        if not k.any(mask):
            print("Found no favorable eigendirection to split, splitting will stop.")
            if save_model:
                models.save_DAN(model, beta_reg, number_memories, split)
            break
        
        init.split_memories(model, mask, 0.25)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0] #@ model.get_DAN_layer(1).get_weights()[1]
    g_cur = model.get_DAN_layer(2).get_weights()[0]
    
    w_plot = w_cur[:, : np.minimum(25, number_memories).astype("int")].T
    g_plot = g_cur[: np.minimum(25, number_memories).astype("int")].T
    
    res_plot.plot_images(w_plot)
    
    res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

models.compile_memorization_phase(model)
number_memories = max_number_memories
number_splits = np.ceil(np.log2(max_number_memories / init_number_memories)).astype("int")

model = models.load_DAN(beta_reg, number_memories, number_splits)

# Get hard test labels and predictions
y_hard = np.argmax(y_test, axis = 1)
y_pred = np.argmax(model(x_test), axis = 1)

model.evaluate(tf.convert_to_tensor(x_test, dtype = "float32"),
               tf.convert_to_tensor(y_test, dtype = "float32"))

w = model.get_DAN_layer(1).get_weights()[0]
g = model.get_DAN_layer(2).get_weights()[0]

res_test.first_neighbor_fidelity(x_test, y_hard, y_pred, w, g)

w_plot = w[:, : 25].T
g_plot = g[: 25].T
res_plot.plot_images(w_plot)
res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

res_test.sort_memories(model, w)

### Fig. 4 and Figs. 7, 8, 9 and 10 of Appendix J: weights learned with unsupervised learning.

In [ ]:
# Seed of random number generation for reproducibility
seed = 2
np_rng = np.random.default_rng(seed)
np_seed, tf_seed = np_rng.integers(0, 2**63, size = 2)
np_rng = np.random.default_rng(np_seed)
tf.random.set_seed(tf_seed)

# Dataset and label softening
dataset = mnist
softening = 0.01

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_patch_data(dataset, normalize_data = True)

# Loss type (supervised or unsupervised)
loss = "unsupervised"

number_features = x_train.shape[1]
number_classes = 10
batch_size = 1000

# Number of epochs of the memorization and splitting phases
number_memorization_epochs = 10
number_splitting_epochs = 0

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 100

# Maximum number of memories, initial number of memories
# and maximum number of splits of splitting steepest descent
max_number_memories = 100
init_number_memories = 100
number_splits = 0

# Initial inverse temperature and inverse temperature regularization
beta_init = 0.01
beta_reg = 0.6

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = None

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
counts_y = np.array([110, 132, 121, 101, 105, 108, 104, 102, 112, 125])
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

# Name of the model and what to record (see DAN_code.models.train_DAN for record options)
name = "DAN"
record = None

if record == "movies":
    try:
        fname = "./Data/Weights/%s_w_with_beta_reg=%s_and_for_movies.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass
    try:
        fname = "./Data/Weights/%s_g_with_beta_reg=%s_and_for_movies.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass

if record == "beta":
    try:
        fname = "./Data/Weights/%s_beta_with_beta_reg=%s.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass

model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features,
                        init_number_memories, max_number_memories, number_classes,
                        number_constraint_iterations, learning_rate, momentum,
                        prior_y, normalize_online = normalize_online)

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

number_memories = init_number_memories
for split in range(number_splits + 1):
    
    model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                             training_phase = "memorization", record = record, verbose = True)
    
    number_memories = model.get_DAN_layer(1).output_size
    max_number_memories = model.get_DAN_layer(1).max_output_size
    
    if number_memories == max_number_memories:
        print("Maximum number of memories reached, cannot split any further.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    elif split == number_splits:
        print("Maximum number of splits reached, splitting will stop.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    else:
        model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                                 training_phase = "splitting", record = None, verbose = True)
        
        max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
        mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                      max_eigval, batch_size, adjust = True)
        
        if not k.any(mask):
            print("Found no favorable eigendirection to split, splitting will stop.")
            if save_model:
                models.save_DAN(model, beta_reg, number_memories, split)
            break
        
        init.split_memories(model, mask, 0.25)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0]
    g_cur = model.get_DAN_layer(2).get_weights()[0]
    
    w_plot = w_cur[:, : np.minimum(25, number_memories).astype("int")].T
    g_plot = g_cur[: np.minimum(25, number_memories).astype("int")].T
    
    res_plot.plot_images(w_plot)
    
    res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

models.compile_memorization_phase(model)
number_memories = max_number_memories
number_splits = np.ceil(np.log2(max_number_memories / init_number_memories)).astype("int")

model = models.load_DAN(beta_reg, number_memories, number_splits)

w = model.get_DAN_layer(1).get_weights()[0]
g = model.get_DAN_layer(2).get_weights()[0]

w_plot = w[:, : 25].T
g_plot = g[: 25].T
res_plot.plot_images(w_plot)
res_plot.plot_labels(np.argmax(model.predict(w_plot), axis = 1), g_plot)

res_test.sort_memories(model, w)

del model

### Fig. 5, part 1: record weights to plot learning dynamics tree.

Weights to plot the learning dynamics tree are recorded by running the code cells below. The learning dynamics tree is plotted in the umap.ipynb notebook using the recorded weights.

Record weights as they are learned without splitting steepest descent.

In [ ]:
# Seed of random number generation for reproducibility
seed = 2
tf.random.set_seed(seed)

# Dataset and label softening
dataset = mnist
softening = 0.1

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_flat_data(dataset, normalize_data = True, softening = softening)
x_train = x_train[: 1000]
y_train = y_train[: 1000]

# Loss type (supervised or unsupervised)
loss = "supervised"

number_features = x_train.shape[1]
number_classes = y_train.shape[1]
batch_size = 100

# Number of epochs of the memorization and splitting phases
number_memorization_epochs = 600
number_splitting_epochs = 0

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 10

# Maximum number of memories, initial number of memories
# and maximum number of splits of splitting steepest descent
max_number_memories = 400
init_number_memories = 400
number_splits = 0

# Initial inverse temperature and inverse temperature regularization
beta_init = 32.
beta_reg = 0.25

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = 1

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
y_hard = np.argmax(y_train, axis = 1)
counts_y = np.bincount(y_hard)
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

# Name of the model and what to record (see DAN_code.models.train_DAN for record options)
name = "DAN"
record = "weights_without_splitting"

if record[: 7] == "weights":
    try:
        fname = "./Data/Weights/%s_w_with_beta_reg=%s_and_%s.npy" % (name, str(beta_reg), record[8 :])
        os.remove(fname)
    except OSError:
        pass
    try:
        fname = "./Data/Weights/%s_g_with_beta_reg=%s_and_%s.npy" % (name, str(beta_reg), record[8 :])
        os.remove(fname)
    except OSError:
        pass

if record == "beta":
    try:
        fname = "./Data/Weights/%s_beta_with_beta_reg=%s.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass
    
model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features, init_number_memories, max_number_memories,
                        number_classes, number_constraint_iterations, learning_rate, momentum,
                        prior_y, normalize_online = normalize_online)

w = model.get_DAN_layer(1).get_weights()

w_init = np.mean(x_train, axis = 0, keepdims = True).T
w_init = w_init / np.sum(w_init**2, keepdims = True)**(1/2)
w_init = np.tile(w_init, reps = (1, init_number_memories))

w_init = init.random_vmf(w_init, eps**(-3/4))

w[0][:, : init_number_memories] = w_init

model.get_DAN_layer(1).set_weights(w)

w = model.get_DAN_layer(1).get_weights()[0]

w_plot = w[:, : 25].T

res_plot.plot_images(w_plot)

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

start_time = time.perf_counter()
for split in range(number_splits + 1):
    
    model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                             training_phase = "memorization", record = record, verbose = True, validation_split = 0)
    
    number_memories = model.get_DAN_layer(1).output_size
    max_number_memories = model.get_DAN_layer(1).max_output_size
    if number_memories == max_number_memories:
        print("Maximum number of memories reached, cannot split any further.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    elif split == number_splits:
        print("Maximum number of splits reached, splitting will stop.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    else:
        model = models.train_DAN(x_train, y_train, model, number_splitting_epochs, batch_size, patience,
                                 training_phase = "splitting", record = None, verbose = True, validation_split = 0)
        
        max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
        mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                      max_eigval, batch_size, adjust = True)
        
        if not k.any(mask):
            print("Found no favorable eigendirection to split, splitting will stop.")
            if beta_reg:
                models.save_DAN(model, beta_reg, number_memories, split)
            break
        
        init.split_memories(model, mask, 0.25)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0]
    g_cur = model.get_DAN_layer(2).get_weights()[0]
    
    w_plot = w_cur[:, : 25].T
    g_plot = g_cur[: 25].T
    
    res_plot.plot_images(w_plot)
    
    y_hard = np.argmax(model.predict(w_plot), axis = 1)
    res_plot.plot_labels(y_hard, g_plot)

Record weights as they are learned with splitting steepest descent.

In [ ]:
# Seed of random number generation for reproducibility
seed = 2
tf.random.set_seed(seed)

# Dataset and label softening
dataset = mnist
softening = 0.1

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_flat_data(dataset, normalize_data = True, softening = softening)
x_train = x_train[: 1000]
y_train = y_train[: 1000]

# Loss type (supervised or unsupervised)
loss = "supervised"

number_features = x_train.shape[1]
number_classes = y_train.shape[1]
batch_size = 100

# Number of epochs of the memorization and splitting phases
number_memorization_epochs = 300
number_splitting_epochs = 300

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 10

# Maximum number of memories, initial number of memories
# and maximum number of splits of splitting steepest descent
max_number_memories = 400
init_number_memories = 100
number_splits = 14

# Initial inverse temperature and inverse temperature regularization
beta_init = 32.
beta_reg = 0.25

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = 1

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
y_hard = np.argmax(y_train, axis = 1)
counts_y = np.bincount(y_hard)
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

# Name of the model and what to record (see DAN_code.models.train_DAN for record options)
name = "DAN"
record = "weights_with_splitting"

if record[: 7] == "weights":
    try:
        fname = "./Data/Weights/%s_w_with_beta_reg=%s_and_%s.npy" % (name, str(beta_reg), record[8 :])
        os.remove(fname)
    except OSError:
        pass
    try:
        fname = "./Data/Weights/%s_g_with_beta_reg=%s_and_%s.npy" % (name, str(beta_reg), record[8 :])
        os.remove(fname)
    except OSError:
        pass

if record == "beta":
    try:
        fname = "./Data/Weights/%s_beta_with_beta_reg=%s.npy" % (name, str(beta_reg))
        os.remove(fname)
    except OSError:
        pass
    
model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features, init_number_memories, max_number_memories,
                        number_classes, number_constraint_iterations, learning_rate, momentum,
                        prior_y, normalize_online = normalize_online)

w = model.get_DAN_layer(1).get_weights()

w_init = np.mean(x_train, axis = 0, keepdims = True).T
w_init = w_init / np.sum(w_init**2, keepdims = True)**(1/2)
w_init = np.tile(w_init, reps = (1, init_number_memories))

w_init = init.random_vmf(w_init, eps**(-3/4))

w[0][:, : init_number_memories] = w_init

model.get_DAN_layer(1).set_weights(w)

w = model.get_DAN_layer(1).get_weights()[0]

w_plot = w[:, : 25].T

res_plot.plot_images(w_plot)

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

start_time = time.perf_counter()
for split in range(number_splits + 1):
    
    model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                             training_phase = "memorization", record = record, verbose = True, validation_split = 0)
    
    number_memories = model.get_DAN_layer(1).output_size
    max_number_memories = model.get_DAN_layer(1).max_output_size
    if number_memories == max_number_memories:
        print("Maximum number of memories reached, cannot split any further.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    elif split == number_splits:
        print("Maximum number of splits reached, splitting will stop.")
        if save_model:
            models.save_DAN(model, beta_reg, number_memories, split)
        break
    
    else:
        model = models.train_DAN(x_train, y_train, model, number_splitting_epochs, batch_size, patience,
                                 training_phase = "splitting", record = None, verbose = True, validation_split = 0)
        
        max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
        mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                      max_eigval, batch_size, adjust = True)
        
        if not k.any(mask):
            print("Found no favorable eigendirection to split, splitting will stop.")
            if save_model:
                models.save_DAN(model, beta_reg, number_memories, split)
            break
        
        init.split_memories(model, mask, 0.25)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0]
    g_cur = model.get_DAN_layer(2).get_weights()[0]
    
    w_plot = w_cur[:, : 25].T
    g_plot = g_cur[: 25].T
    
    res_plot.plot_images(w_plot)
    
    y_hard = np.argmax(model.predict(w_plot), axis = 1)
    res_plot.plot_labels(y_hard, g_plot)

### Fig. 6: training time with and without splitting steepest descent.

User-defined parameters common to training with and without splitting steepest descent.

Run this code cell first, then run either of the next two code cells to set the parameters for training with or without splitting steepest descent.

The largest networks took a bit more than 24 hours to train in our experiments.

In [ ]:
# Dataset and label softening
dataset = mnist
softening = 0.1

# Import data and reshape to vectors
x_train, y_train, x_test, y_test = data_proc.prepare_flat_data(dataset, normalize_data = True, softening = softening)

# Loss type (supervised or unsupervised)
loss = "supervised"

number_features = x_train.shape[1]
number_classes = y_train.shape[1]
batch_size = 1000

# Number of iterations of the Sinkhorn-Knopp algorithm to constrain the class weights
number_constraint_iterations = 10

# Initial inverse temperature and inverse temperature regularization
beta_init = 1.
beta_reg = 0.25

# Optimizer hyperparameters and patience for early stopping
learning_rate = 0.1
momentum = 0.99
patience = 1

# Splitting steepest descent hyperparameters
max_fraction_eigvals = 1.
max_eigval = -0.001

# Boolean settings
normalize_online = False
save_model = True

# Calculate the class prior
y_hard = np.argmax(y_train, axis = 1)
counts_y = np.bincount(y_hard)
counts_y = np.concatenate([counts_y, np.array([0])])
prior_y = (1 - softening) * counts_y / np.sum(counts_y) + softening * 1/(number_classes + 1)

x_train = tf.convert_to_tensor(x_train, dtype = "float32")
y_train = tf.convert_to_tensor(y_train, dtype = "float32")

Parameter values used for training with splitting steepest descent.

Change max_number_memories to choose which data point of Fig. 6 to reproduce.

In [ ]:
# Seed of random number generation for reproducibility
seed = 2
np_rng = np.random.default_rng(seed)

max_number_memories = 1000
#max_number_memories = 1125
#max_number_memories = 1250
#max_number_memories = 1375
#max_number_memories = 1500
#max_number_memories = 1625
#max_number_memories = 1750
#max_number_memories = 1875
#max_number_memories = 2000

init_number_memories = 100

number_memorization_epochs = 100
number_splitting_epochs = 1

number_splits = 14

Parameter values used for training without splitting steepest descent.

Change max_number_memories to choose which data point of Fig. 6 to reproduce.

In [ ]:
# Seed of random number generation for reproducibility
seed = 2
np_rng = np.random.default_rng(seed)

max_number_memories = 1000
#max_number_memories = 1250
#max_number_memories = 1500
#max_number_memories = 1750
#max_number_memories = 2000

init_number_memories = max_number_memories

number_memorization_epochs = 100 + max_number_memories // 10
number_splitting_epochs = 0

number_splits = 0

Run this code cell to estimate the accuracy and training time for the parameters set in the cells above.

Estimate the bare training time as a minimum over many runs as described in the paper.

In some cases, the training times obtained for some value of max_number_memories may be significantly larger than expected from Fig. 6. In such cases, rerun the training with the same value of max_number_memories, then take the component-wise minimum of the training times over the runs to mitigate the noise due to background processes, as explained in the paper.


In [ ]:
number_repeats = 10

run_times = []
accuracies = []

for _ in range(number_repeats):
    run_time = np.inf
    np_seed, tf_seed = np_rng.integers(0, 2**63, size = 2)
    np_rng = np.random.default_rng(np_seed)
    
    for _ in range(number_repeats):
        tf.random.set_seed(tf_seed)
        
        model = models.init_DAN(loss, beta_init, beta_reg, softening, number_features,
                                init_number_memories, max_number_memories, number_classes,
                                number_constraint_iterations, learning_rate, momentum,
                                prior_y, normalize_online = normalize_online)

        start_time = time.perf_counter()
        for split in range(number_splits + 1):
            cur_split = split

            model = models.train_DAN(x_train, y_train, model, number_memorization_epochs, batch_size, patience,
                                     training_phase = "memorization", record = None, verbose = False)
            
            number_memories = model.get_DAN_layer(1).output_size
            max_number_memories = model.get_DAN_layer(1).max_output_size

            if number_memories == max_number_memories:
                break

            elif split == number_splits:
                break

            else:
                model = models.train_DAN(x_train, y_train, model, number_splitting_epochs, batch_size, patience,
                                         training_phase = "splitting", record = None, verbose = False)

                max_number_eigvals = int(np.rint(max_fraction_eigvals * number_memories))
                mask = models.calc_split_mask(x_train, y_train, model, max_number_eigvals,
                                              max_eigval, batch_size, adjust = True)

                if not k.any(mask):
                    break

                init.split_memories(model, mask, 0.25)

        end_time = time.perf_counter()

        run_time = np.minimum(run_time, end_time - start_time)
    
    print(run_time)
    run_times.append(run_time)
    
    w_cur = model.get_DAN_layer(1).get_weights()[0]
    g_cur = model.get_DAN_layer(2).get_weights()[0]

    w_plot = w_cur[:, : 25].T
    g_plot = g_cur[: 25].T

    res_plot.plot_images(w_plot)

    y_hard = np.argmax(model.predict(w_plot), axis = 1)
    res_plot.plot_labels(y_hard, g_plot)

    models.compile_memorization_phase(model)

    accuracy = model.evaluate(x_test, y_test)[1]
    accuracies.append(accuracy)

number_splits = cur_split
with open("./Data/Performance/DAN_accuracy_and_run_time_with_beta_reg=%s_and_%s_memories_for_%s_splits.npy"
          % (str(beta_reg), str(number_memories), str(number_splits)), "wb") as f:
    np.save(f, np.array(run_times))
    np.save(f, np.array(accuracies))

Plotting.

Assumes that the accuracy and training time were already calculated and saved using the code cells above.

In [ ]:
beta_reg = 0.25
number_memories_with_splits_range = [1000, 1125, 1250, 1375, 1500, 1625, 1750, 1875, 2000]
number_memories_without_splits_range = [1000, 1250, 1500, 1750, 2000]
number_splits_range = [4, 4, 4, 4, 4, 5, 5, 5, 5]

res_plot.plot_accuracy_and_runtime(beta_reg, number_memories_with_splits_range,
                                   number_memories_without_splits_range, number_splits_range,
                                   filename = "performance_vs_number_memories_with_errobars")